In [1]:
import numpy as np
# import seaborn as sns
import pandas as pd
import os.path

import matplotlib.pyplot as plt

import tifffile 
import czifile

from skimage import transform
from scipy import ndimage

import random 
import math

In [ ]:
patch_size = 16
major_ch_array = [0,1,2,3]
ctr_or_y = 'y'
ctrl_y_str = ctr_or_y+'_allch'

seg_folder_str = 'code_org_20250820_seg'
group_str = ctrl_y_str[:-6]+'_patches_gridonly_wholecell_ps16_allch'



In [3]:
start_ind = 0
end_ind = 51

if(ctr_or_y=='ctrl'):
    image_folder = '/mnt/d/lding/FA/data/FA_ML_Annabel_20250217/031125data/Control'
if(ctr_or_y=='y'):
    image_folder = '/mnt/d/lding/FA/data/FA_ML_Annabel_20250217/031125data/Ycomp'
        

In [4]:

cell_mask_folder = '/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/y_ch1_major/'+seg_folder_str+'/mask'

front_mask_dir = '/mnt/d/lding/FA/data/FA_ML_Annabel_20250217/031125data/frontmasks'

In [5]:
half_ps = int(patch_size/2)
half_half_ps = int(patch_size/4)
double_ps = patch_size*2
double_double_ps = patch_size*4

In [6]:
def rotate_coor(x_i,y_i,x_c,y_c,rotate_angle):
 
    rotate_angle = rotate_angle*np.pi/180
 
    x_o = (x_i-x_c)*math.cos(rotate_angle) - (2*y_c-y_i-y_c)*math.sin(rotate_angle) +x_c
    y_o = -(x_i-x_c)*math.sin(rotate_angle) - (2*y_c-y_i-y_c)*math.cos(rotate_angle) +(2*y_c-y_c)

    return([x_o,y_o])

In [7]:
from datetime import datetime
now = datetime.now()

time_str = now.strftime("%Y%m%d_%H%M")

In [8]:
mask_ratio = 0.4
movie_partitioned_data_dir = '/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/'\
    +ctrl_y_str+'/'+group_str+ '/tiff_patches'+str(patch_size)+'_40p_'+time_str
movie_plot_dir = '/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/' \
    +ctrl_y_str+'/'+group_str+ '/plot_patches'+str(patch_size)+'_40p_'+time_str
os.makedirs(movie_partitioned_data_dir,exist_ok=True)
os.makedirs(movie_plot_dir,exist_ok=True)


In [ ]:
data_prep_record = pd.DataFrame(columns={'image_folder','filename','filenameID','x_c','y_c','rand_angle','rand_tx','rand_ty',
                            'x_corner1','x_corner2','x_corner3','x_corner4','y_corner1','y_corner2','y_corner3','y_corner4',
                            'movie_partitioned_data_dir','crop_img_filename','movie_plot_dir','plot_filename', 'dist_cell_edge'})

In [10]:
from skimage.feature import peak_local_max



In [11]:
def image_padding(input_img, pad_size,value):
    output_img = np.zeros([input_img.shape[0]+pad_size*2,input_img.shape[1]+pad_size*2]) + value
    output_img[pad_size:input_img.shape[0]+pad_size,pad_size:input_img.shape[1]+pad_size] = input_img
    return output_img

In [12]:
filenames = [x for x in os.listdir(image_folder) if os.path.isfile(os.path.join(image_folder, x)) and ('.czi' in x)]

debug_flag = 0

for major_ch in major_ch_array:
    ch_movie_partitioned_data_dir = os.path.join(movie_partitioned_data_dir,'ch'+str(major_ch))
    os.makedirs(ch_movie_partitioned_data_dir,exist_ok=True)

for major_ch in major_ch_array:
    ch_movie_plot_dir = os.path.join(movie_plot_dir,'ch'+str(major_ch))
    os.makedirs(ch_movie_plot_dir,exist_ok=True)

    

for filenameID in range(start_ind, min(end_ind,len(filenames))):
# for filenameID in range(0,1):
    if debug_flag ==1:
                break
    filename = filenames[filenameID]

    for major_ch in major_ch_array:
        train_img = czifile.imread(os.path.join(image_folder, filename)).squeeze()[major_ch,:,:].astype(float)/255/255
        train_seg = tifffile.imread(os.path.join(cell_mask_folder, "cell_mask_"+filename+".tif")).squeeze().astype(float)
        train_img = image_padding(train_img, 64, np.mean(train_img))
        train_seg = image_padding(train_seg, 64, 0)

        fig_accu, ax_accu = plt.subplots(1,2, figsize=(15.6,7.8), dpi=256, facecolor='w', edgecolor='k')
        ax_accu[0].imshow(train_img, cmap=plt.cm.gray,vmax=1,vmin=0)
        ax_accu[1].imshow(train_seg, cmap=plt.cm.gray,vmax=1,vmin=0)    
        
        x_num = int(np.floor(train_img.shape[1]/patch_size))
        y_num = int(np.floor(train_img.shape[0]/patch_size))
        
        x_0 = int((train_img.shape[1] - x_num*patch_size)/2+ (0.0)*patch_size)
        y_0 = int((train_img.shape[0] - y_num*patch_size)/2+ (0.0)*patch_size)
        
                
        for x_i in range(x_num):
            if debug_flag ==1:
                    break
            for y_i in range(x_num):

                if debug_flag ==1:
                    break

                y_c = int(y_0+(y_i-0.5)*patch_size)
                x_c = int(x_0+(x_i-0.5)*patch_size)

                y_left = y_c - double_ps
                x_left = x_c - double_ps

                y_right = y_c + double_ps
                x_right = x_c + double_ps


                if y_left < 0 or x_left < 0 or y_right >= train_img.shape[0] or x_right >= train_img.shape[1]:
                    continue

                patch_img = train_img[y_left:y_right,x_left:x_right]
                patch_seg = train_seg[y_left:y_right,x_left:x_right]
                
                if((patch_seg.mean())<mask_ratio/64):
                    continue
                
                rand_tx = 0
                rand_ty = 0

                # crop around the local maximum first
                cx_left_1 = patch_size-rand_tx
                cx_right_1 = double_ps+patch_size-rand_tx
                cy_up_1 = patch_size-rand_ty
                cy_down_1 = double_ps+patch_size-rand_ty

                big_crop_patch_img = patch_img[cy_up_1:cy_down_1,
                                        cx_left_1:cx_right_1]
                big_crop_patch_seg = patch_seg[cy_up_1:cy_down_1,
                                        cx_left_1:cx_right_1]
                
                first_crop_x = np.array([cx_left_1, cx_left_1, cx_right_1, cx_right_1, cx_left_1]) + x_left
                first_crop_y = np.array([cy_up_1, cy_down_1, cy_down_1, cy_up_1, cy_up_1]) + y_left            


                if((big_crop_patch_seg.mean())<mask_ratio/32):
                    continue

                rand_angle = random.random() * 0
                rotated_patch_img = transform.rotate(big_crop_patch_img, rand_angle, resize=False, center=None, order=None, mode='constant', cval=0, clip=True)
                rotated_patch_seg = transform.rotate(big_crop_patch_seg, rand_angle, resize=False, center=None, order=None, mode='constant', cval=0, clip=True)
            
                cx_left_2 = half_ps
                cx_right_2 = patch_size+half_ps
                cy_up_2 = half_ps
                cy_down_2 = patch_size+half_ps

                crop_patch_img = rotated_patch_img[cy_up_2:cy_down_2,
                                        cx_left_2:cx_right_2]
                crop_patch_seg = rotated_patch_seg[cy_up_2:cy_down_2,
                                        cx_left_2:cx_right_2]

                if((crop_patch_seg.mean())<mask_ratio):
                    continue

                crop_img_filename = 'ch'+str(major_ch)+'_image'+str(filenameID).zfill(4)+'x'+str(x_c).zfill(4)+'y'+str(y_c).zfill(4)+'ps'+str(patch_size)+'.tif'
                
                ch_movie_partitioned_data_dir = os.path.join(movie_partitioned_data_dir,'ch'+str(major_ch))    

                
                tifffile.imwrite(
                    os.path.join(ch_movie_partitioned_data_dir,crop_img_filename),
                    crop_patch_img.astype(np.float32),
                    imagej=True,              # Write ImageJ metadata block
                    metadata={'axes': 'YX'}   # Or 'TYX', 'ZYX', etc. depending on shape
                )

                X_TransRotCrop = np.array([cx_left_2, cx_left_2, cx_right_2, cx_right_2, cx_left_2])
                Y_TransRotCrop = np.array([cy_up_2, cy_down_2, cy_down_2, cy_up_2, cy_up_2])

                [X_invrotate_patch, Y_invrotate_patch] = rotate_coor(X_TransRotCrop,Y_TransRotCrop,patch_size,patch_size,-rand_angle)

                X_bigger_patch = X_invrotate_patch + x_left + cx_left_1
                Y_bigger_patch = Y_invrotate_patch + y_left + cy_up_1


                plot_filename = 'plot_grid_t'+str(filenameID).zfill(4)+'_xc'+str(x_c)+'_yc'+str(y_c)+ '.png'
                
                ax_accu[0].plot(X_bigger_patch, Y_bigger_patch,color='green')
                ax_accu[1].plot(X_bigger_patch, Y_bigger_patch,color='green')
                
                s = pd.Series([image_folder, filename, filenameID, x_c,y_c,rand_angle,rand_tx,rand_ty,
                    X_bigger_patch[0],X_bigger_patch[1],X_bigger_patch[2],X_bigger_patch[3],
                    Y_bigger_patch[0],Y_bigger_patch[1],Y_bigger_patch[2],Y_bigger_patch[3],
                    movie_partitioned_data_dir,crop_img_filename,movie_plot_dir,plot_filename],
                    index=['image_folder','filename','filenameID','x_c','y_c','rand_angle','rand_tx','rand_ty',
                                            'x_corner1','x_corner2','x_corner3','x_corner4','y_corner1','y_corner2','y_corner3','y_corner4',
                                            'movie_partitioned_data_dir','crop_img_filename','movie_plot_dir','plot_filename'])
                
                data_prep_record = data_prep_record.append(s,ignore_index=True)

        ch_movie_plot_dir = os.path.join(movie_plot_dir,'ch'+str(major_ch))
    
        data_prep_record.to_csv(os.path.join(ch_movie_plot_dir, 'vin_set_'+'ch'+str(major_ch)+'_data_prep_record_'+str(filenameID)+'_t'+str(filenameID)+'.csv'))

        fig_accu.savefig(os.path.join(ch_movie_plot_dir,'grid_t'+str(filenameID).zfill(4)+'.png'))   
        plt.close(fig_accu)   
            

In [ ]:
# normalize with the group profile

profile = pd.read_csv('/mnt/d/lding/FA/data/FA_ML_Annabel_20250217/031125data/Control/control_channel_profiles.csv')
profile = pd.read_csv('/mnt/d/lding/FA/data/FA_ML_Annabel_20250217/031125data/YComp/ycomp_channel_profiles.csv')

In [14]:
ch0_p01 = profile['ch0_percentile_1'][0]/65535.0
ch0_p99 = profile['ch0_percentile_99'][0]/65535.0
ch1_p01 = profile['ch1_percentile_1'][0]/65535.0
ch1_p99 = profile['ch1_percentile_99'][0]/65535.0
ch3_p01 = profile['ch3_percentile_1'][0]/65535.0
ch3_p99 = profile['ch3_percentile_99'][0]/65535.0

In [15]:
ch0_p99/65535.0

7.763276235288809e-06

In [16]:
ch1_p99

0.5548180361638819

In [17]:
ch3_p01

0.0002136263065537499

In [18]:
def norm_profile(input_img,ch0_p01,ch0_p99):
    output_img = (input_img - ch0_p01)/(ch0_p99-ch0_p01)
    output_img[output_img<0]=0
    output_img[output_img>1]=1
    return output_img

In [20]:
ch013_folder = os.path.join(movie_partitioned_data_dir,'ch013')
ch013_normed_folder = os.path.join(movie_partitioned_data_dir,'ch013_normed')
    
os.makedirs(ch013_folder,exist_ok=True)
os.makedirs(ch013_normed_folder,exist_ok=True)

ch0_folder = os.path.join(movie_partitioned_data_dir,'ch0')
ch1_folder = os.path.join(movie_partitioned_data_dir,'ch1')
ch3_folder = os.path.join(movie_partitioned_data_dir,'ch3')

filenames = [x for x in os.listdir(ch0_folder) if os.path.isfile(os.path.join(ch0_folder, x)) and ('.tif' in x)]

for filenameID in range(0,len(filenames)):
    filename = filenames[filenameID]
    # print(filename)

    patch_ch0 = tifffile.imread(os.path.join(ch0_folder, filename))
    patch_ch1 = tifffile.imread(os.path.join(ch1_folder, filename.replace('ch0','ch1')))
    patch_ch3 = tifffile.imread(os.path.join(ch3_folder, filename.replace('ch0','ch3')))

    patch_ch013 = np.stack([patch_ch0, patch_ch1, patch_ch3], axis=0)     

    crop_img_filename = 'ch013'+str(major_ch)+'_image'+str(filenameID).zfill(4)+'_x'+str(x_c).zfill(4)+'y'+str(y_c).zfill(4)+'ps'+str(patch_size)+'.tif'
    # print(crop_img_filename)
    tifffile.imwrite(
        os.path.join(ch013_folder,crop_img_filename),
        patch_ch013.astype(np.float32),
        imagej=True,              # Write ImageJ metadata block
        metadata={'axes': 'CYX'}   # Or 'TYX', 'ZYX', etc. depending on shape
    )

    patch_ch0_normed = norm_profile(patch_ch0,ch0_p01,ch0_p99)
    patch_ch1_normed = norm_profile(patch_ch1,ch1_p01,ch1_p99)
    patch_ch3_normed = norm_profile(patch_ch3,ch3_p01,ch3_p99)
 

    patch_ch013_normed = np.stack([patch_ch0_normed, patch_ch1_normed, patch_ch3_normed], axis=0)     

    
    crop_img_filename = 'ch013'+str(major_ch)+'_normed_image'+str(filenameID).zfill(4)+'_x'+str(x_c).zfill(4)+'y'+str(y_c).zfill(4)+'ps'+str(patch_size)+'.tif'
    # print(crop_img_filename)
    tifffile.imwrite(
        os.path.join(ch013_normed_folder,crop_img_filename),
        patch_ch013_normed.astype(np.float32),
        imagej=True,              # Write ImageJ metadata block
        metadata={'axes': 'CYX'}   # Or 'TYX', 'ZYX', etc. depending on shape
    )

            